# Apex Lark — Day 09: Exploratory Data Analysis & Business Intelligence
**Dataset:** `DOC-20261102-WA0001.csv`

This notebook follows the supplied Day 09 brief: dataset understanding, EDA questions, descriptive statistics, IQR outliers, correlation analysis, eight visualizations, and business recommendations.

> Data handling: fully blank rows and exact duplicate rows are excluded from analytical calculations. Categorical labels are normalized for grouping. Missing values are not silently imputed. Implausible values are flagged for review rather than automatically changing sales or profit values.

In [ ]:
# 1. Import libraries and load data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("DOC-20261102-WA0001.csv")
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

print("Raw shape:", df.shape)
display(df.head(10))
display(df.tail(10))
df.info()
display(df.describe(include="all").T)
print("Rows and columns:", df.shape)
print("Memory usage (bytes):", df.memory_usage(deep=True).sum())
print("Missing values:\n", df.isna().sum())
print("Exact duplicate rows:", df.duplicated().sum())

## 2. Prepare a consistent analysis dataset

In [ ]:
# Remove fully blank rows and exact duplicate records
data = df.dropna(how="all").drop_duplicates().copy()

# Normalize text labels so e.g. 'male', 'MALE' and 'M' group together
for col in ["gender", "region", "city", "product_category", "product_name",
            "payment_method", "order_status"]:
    data[col] = data[col].astype("string").str.strip()

data["gender"] = data["gender"].str.lower().replace({"m": "male", "f": "female"}).str.title()
data["order_status"] = data["order_status"].str.title()
data["return_flag"] = data["return_flag"].astype("boolean")
data["month"] = data["order_date"].dt.to_period("M").astype(str)

print("Analysis shape:", data.shape)
print("Unique orders:", data["order_id"].nunique())
print("Unique customers:", data["customer_id"].nunique())
print("Date range:", data["order_date"].min(), "to", data["order_date"].max())

## 3. Exploratory Data Analysis

In [ ]:
# Product with highest profit
display(data.groupby("product_name")["profit"].sum().sort_values(ascending=False).head(10))

# Category with highest sales
display(data.groupby("product_category")["sales_amount"].sum().sort_values(ascending=False))

# Region with highest revenue
display(data.groupby("region")["sales_amount"].sum().sort_values(ascending=False))

# City with the most orders
display(data.groupby("city")["order_id"].nunique().sort_values(ascending=False).head(10))

# Customer segment represented by gender field with highest profit
display(data.groupby("gender")["profit"].sum().sort_values(ascending=False))

# Month with highest sales
monthly_sales = data.groupby(data["order_date"].dt.to_period("M"))["sales_amount"].sum()
display(monthly_sales.sort_values(ascending=False).head(10))

# Average order value: total order sales divided by unique order IDs
order_totals = data.groupby("order_id")["sales_amount"].sum()
print("Average order value:", order_totals.mean())

# Top 10 customers by total sales
display(data.groupby(["customer_id", "customer_name"])["sales_amount"]
        .sum().sort_values(ascending=False).head(10))

## 4. Statistical Analysis and IQR Outliers

In [ ]:
numeric = data.select_dtypes(include="number")
display(numeric.agg(["mean", "median", lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan,
                     "std", "var"]).T.rename(columns={"<lambda>": "mode"}))

quartiles = data["sales_amount"].quantile([0.25, 0.50, 0.75])
q1, q3 = quartiles.loc[0.25], quartiles.loc[0.75]
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
sales_outliers = data[(data["sales_amount"] < lower_bound) |
                      (data["sales_amount"] > upper_bound)]
print("Sales Q1:", q1, "Median:", quartiles.loc[0.50], "Q3:", q3)
print("IQR:", iqr, "Lower bound:", lower_bound, "Upper bound:", upper_bound)
print("Potential sales outliers:", len(sales_outliers))
display(sales_outliers[["order_id", "sales_amount", "profit", "product_name"]].head(20))

# Quality flags: inspect these before using customer/delivery fields for decisions
print("Age outside 0–100:", ((data.age < 0) | (data.age > 100)).sum())
print("Quantity <= 0:", (data.quantity <= 0).sum())
print("Shipping days outside 0–60:", ((data.days_to_ship < 0) | (data.days_to_ship > 60)).sum())

## 5. Correlation Analysis

In [ ]:
corr = numeric.corr()
display(corr)
plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

# Find strongest positive and negative unique pairs
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
pairs = upper.stack().sort_values()
print("Strongest negative pair:", pairs.head(1))
print("Strongest positive pair:", pairs.tail(1))

## 6. Data Visualizations

In [ ]:
# Monthly sales trend
monthly_sales.sort_index().plot(figsize=(12, 4), marker="o", title="Monthly Sales Trend")
plt.xlabel("Month"); plt.ylabel("Sales Amount"); plt.xticks(rotation=60); plt.tight_layout(); plt.show()

# Sales by region
data.groupby("region")["sales_amount"].sum().sort_values().plot(kind="barh", title="Sales by Region")
plt.xlabel("Sales Amount"); plt.tight_layout(); plt.show()

# Profit by category
data.groupby("product_category")["profit"].sum().sort_values().plot(kind="bar", title="Profit by Category")
plt.xlabel("Category"); plt.ylabel("Profit"); plt.xticks(rotation=35); plt.tight_layout(); plt.show()

# Sales distribution (cap display at 99th percentile to make the main body visible)
data["sales_amount"].clip(upper=data["sales_amount"].quantile(.99)).hist(bins=30, figsize=(8,4))
plt.title("Sales Distribution (visual capped at 99th percentile)")
plt.xlabel("Sales Amount"); plt.ylabel("Frequency"); plt.tight_layout(); plt.show()

# Profit distribution
data["profit"].plot(kind="box", title="Profit Distribution")
plt.ylabel("Profit"); plt.tight_layout(); plt.show()

# Correlation heatmap
plt.figure(figsize=(11,8)); sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Correlation Heatmap"); plt.tight_layout(); plt.show()

# Top 10 customers
(data.groupby(["customer_id", "customer_name"])["sales_amount"].sum()
 .sort_values().tail(10).plot(kind="barh", figsize=(8,5), title="Top 10 Customers by Sales"))
plt.xlabel("Sales Amount"); plt.tight_layout(); plt.show()

# Category sales pie chart
data.groupby("product_category")["sales_amount"].sum().plot(kind="pie", autopct="%1.1f%%", figsize=(7,7))
plt.title("Product Category Sales"); plt.ylabel(""); plt.tight_layout(); plt.show()

## 7. Business Insights and Recommendations

1. Electronics leads sales and profit in the observed data; maintain availability of high-performing electronics products and monitor margin by SKU.
2. The South region has the highest total sales; evaluate whether inventory and promotional resources are aligned with its demand.
3. The highest-sales month is a useful starting point for seasonal planning, but validate the pattern across multiple years before committing budget.
4. Review unusually large sales/profit observations and IQR outliers to confirm whether they are genuine bulk orders or data-entry issues.
5. Improve data quality: investigate blank records, missing values, inconsistent labels, implausible ages/quantities/shipping days, and return/status inconsistencies before predictive modelling.

**Interpretation caution:** the `gender` field is used as the available customer-segment proxy because the dataset has no separate customer-segment column. Correlation is association, not causation.